# Activity 14: Sentiment Analysis on Real Reviews

Brandon Parks · OIM3641

Amazon Digital Video Games reviews. The dataset stays zipped in the data folder. Models use the existing local Hugging Face cache.

## Setup check

This checks the environment only. It does not load reviews or classify them.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import torch
from transformers import pipeline

torch.set_num_threads(2)
data_path = Path('data/amazon_reviews_us_Digital_Video_Games_v1_00.tsv.zip')
assert data_path.is_file(), 'Open this notebook from the hf-explore folder.'
print('Ready:', data_path.name)
print('Python:', sys.executable)
import time
import gc


OMP: Warning #179: Function Can't set size of /tmp file failed:


Ready: amazon_reviews_us_Digital_Video_Games_v1_00.tsv.zip
Python: /Users/brandonparks/.local/share/classwork-envs/hf-explore/bin/python


## Step 2 Load the data


In [2]:
DATA_PATH = str(data_path)
df = pd.read_csv(DATA_PATH, sep="\t")
print("Default settings:", df.shape)

Default settings: (144724, 15)


**Questions:** How many rows and columns did pandas load?

Pandas loaded 144,724 rows and 15 columns with the default settings.

## Step 3 Load it again correctly


In [3]:
df = pd.read_csv(DATA_PATH, sep="\t", quoting=3)
print("quoting=3:", df.shape)

quoting=3: (145431, 15)


**Questions:** How many rows does each version load? Did pandas warn you? What does this tell you about checking data?

The default version loaded 144,724 rows. With quoting=3, it loaded 145,431 rows, or 707 more. Pandas did not warn about the missing rows. A file loading without an error does not mean it loaded correctly. I would check the row count and a few actual records before using it.

## Step 4 Read some actual reviews


In [4]:
peek = df["review_body"].sample(5, random_state=2134)
for text in peek:
    print(text, "\n")

The &#34;AI&#34; is wrong by cheating, many bugs and control system cheating is not reliable<br />the PC version is penalized 

Is fast secure and very easy you only payed and in less than a minute you can use the code thanks 

This is exactly what it says it is, and I was able to email it as a gift!  Perfect.  Thanks! 

Since there is no way to return downloaded games or even remark on my problem, my only other options is via a review.<br /><br />I have a steam account and am logged in.  I tried installing the game, and it gave me a generic &#34;Trouble Communicating With Gateway&#34; message.  Googling the message returns one web page in German.  Retried three times with the same result.<br /><br />So...  Looks like I'm screwed. 

I bought this crapware online but was not able to install it on my tablet nor my android phone. The amazon android market app \\"seems\\" to crash when installing free or promotional apps. Amazon customer service is non existent. They dont care once they ha

**Questions:** What looks weird in these reviews? Where do you think it came from?

Some reviews contain <br /> tags, &#34; instead of quotation marks, and extra backslashes. These look like leftover HTML and escaped text from the website or the export, rather than something the customer meant to write.

## Step 5 Look at the star ratings


In [5]:
df["star_rating"].value_counts(normalize=True).sort_index()

star_rating
1    0.171697
2    0.053283
3    0.079962
4    0.140314
5    0.554744
Name: proportion, dtype: float64

**Questions:** How are the ratings distributed? Why does that matter when measuring accuracy?

About 17.17% are 1 star, 5.33% are 2 stars, 8.00% are 3 stars, 14.03% are 4 stars, and 55.47% are 5 stars. Five-star reviews are more than half the data. A model could look fairly accurate just by predicting the most common rating, while doing badly on unhappy customers.

## Step 6 Look at the products


In [6]:
df["product_title"].value_counts().head(10)

product_title
Playstation Network Card                                          13642
Xbox Live Subscription                                             7307
Playstation Plus Subscription                                      4712
Xbox Live Gift Card                                                3438
SimCity - Limited Edition                                          3421
Xbox 360 Live Points Card                                          3036
Battlefield 4                                                      1453
Final Fantasy XIV: A Realm Reborn                                  1273
Battlefield 3                                                      1160
Command and Conquer The Ultimate Collection [Online Game Code]      875
Name: count, dtype: int64

**Questions:** What do you notice about the most reviewed games?

A lot of the biggest products are not individual games. They are PlayStation Network cards, Xbox Live subscriptions, PlayStation Plus subscriptions, and gift or points cards. The dataset also includes things like purchases and account services, so not every complaint will be about gameplay.

## Step 7 Clean the text


In [7]:
print("Reviews with <br />:", df["review_body"].str.contains("<br />").sum())
df = df.dropna(subset=["review_body"]).copy()
df["review_body"] = df["review_body"].str.replace("<br />", " ", regex=False)
print("After cleaning:", df["review_body"].str.contains("<br />").sum(), "\n")
for text in df.loc[peek.index, "review_body"]:
    print(text, "\n")

Reviews with <br />: 25699
After cleaning: 0 

The &#34;AI&#34; is wrong by cheating, many bugs and control system cheating is not reliable the PC version is penalized 

Is fast secure and very easy you only payed and in less than a minute you can use the code thanks 

This is exactly what it says it is, and I was able to email it as a gift!  Perfect.  Thanks! 

Since there is no way to return downloaded games or even remark on my problem, my only other options is via a review.  I have a steam account and am logged in.  I tried installing the game, and it gave me a generic &#34;Trouble Communicating With Gateway&#34; message.  Googling the message returns one web page in German.  Retried three times with the same result.  So...  Looks like I'm screwed. 

I bought this crapware online but was not able to install it on my tablet nor my android phone. The amazon android market app \\"seems\\" to crash when installing free or promotional apps. Amazon customer service is non existent. They 

**Questions:** How many reviews had line breaks before and after cleaning? What still looks weird, and might it affect sentiment?

There were 25,699 reviews containing <br /> before cleaning and 0 afterward. The five reviews still contain &#34; and backslashes, plus spelling and grammar mistakes. Those might confuse a model, but the main complaints and positive words are still understandable. Removing line breaks alone does not completely clean the text.

## Step 8 Take a balanced sample


In [8]:
sample = df.groupby("star_rating").sample(n=100, random_state=42).copy()
complaints = df[df["star_rating"] == 1].sample(n=20, random_state=42).copy()
print(len(sample), len(complaints))

500 20


**Questions:** Why take the same number of each star rating instead of 500 random reviews?

Taking 100 of each rating makes sure all five groups are represented equally. A random sample would probably contain mostly five-star reviews and very few two-star reviews. This makes it easier to compare performance across ratings, although the balanced sample does not represent the original rating proportions.

## Step 9 Try one review


In [9]:
classifier = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english", device=-1)
print(sample["star_rating"].iloc[0], sample["review_body"].iloc[0][:200])
classifier(sample["review_body"].iloc[0])

Device set to use cpu


1 Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get 


[{'label': 'NEGATIVE', 'score': 0.9998071789741516}]

**Questions:** What are the result keys, and what does the score mean? Does it agree with the customer's rating?

The result has label and score keys. The score is the model's confidence for its predicted label, not the customer's rating or a guarantee that the answer is correct. This review is rated 1 star and the model predicts NEGATIVE with a score around 0.9998, so they agree.

## Step 10 Classify all 500 reviews


In [10]:
started = time.perf_counter()
results = classifier(sample["review_body"].tolist(), truncation=True, batch_size=8)
elapsed = time.perf_counter() - started
sample["label"] = [r["label"] for r in results]
sample["score"] = [r["score"] for r in results]
print(f"500 reviews: {elapsed:.1f} seconds")
print(f"145,000 reviews at this rate: {elapsed * 145000 / 500 / 3600:.2f} hours")
sample[["star_rating", "label", "score"]].head(10)

500 reviews: 16.7 seconds
145,000 reviews at this rate: 1.35 hours


,star_rating,label,score
40836,1,NEGATIVE,0.999807
85497,1,NEGATIVE,0.990978
124434,1,NEGATIVE,0.998919
118528,1,NEGATIVE,0.999704
13480,1,NEGATIVE,0.999466
47179,1,NEGATIVE,0.999234
130193,1,NEGATIVE,0.999613
126902,1,NEGATIVE,0.999815
58985,1,NEGATIVE,0.999799
22608,1,NEGATIVE,0.999736


**Questions:** How long did it take? Estimate the time for all 145,000. What does truncation=True protect against?

The 500 reviews took about 16.7 seconds on this computer, using batches of 8. At the same rate, 145,000 reviews would take about 4,843 seconds, or 81 minutes (1.35 hours). This is only an estimate because review lengths and computer load vary. truncation=True cuts text that exceeds the model's token limit so a very long review does not cause a length error. It can also cut off useful information at the end.

## Step 11 Create the answer key and score the model


In [11]:
def stars_to_sentiment(stars):
    if stars <= 2:
        return "NEGATIVE"
    if stars >= 4:
        return "POSITIVE"
    return "NEUTRAL"

sample["truth"] = sample["star_rating"].apply(stars_to_sentiment)
graded = sample[sample["truth"] != "NEUTRAL"]
print("Accuracy:", (graded["label"] == graded["truth"]).mean())

Accuracy: 0.8775


**Questions:** Why leave out 3 star reviews? What is the accuracy? Would you trust it in a product?

Three stars can be mixed or neutral, but this model only predicts POSITIVE or NEGATIVE, so those 100 reviews are excluded from this accuracy calculation. It got 351 of the other 400 right, or 87.75%. I would use it for a rough trend, not trust it to assign final ratings without checking mistakes. This small balanced sample is also not the same as testing every real customer review.

## Step 12 Break it down by star rating


In [12]:
pd.crosstab(sample["star_rating"], sample["label"])

label,NEGATIVE,POSITIVE
star_rating,,
1,96,4
2,95,5
3,66,34
4,23,77
5,17,83


**Questions:** Which rating is wrong most often? What happened to the 3 star reviews?

Four-star reviews are wrong most often: 23 out of 100 were called NEGATIVE. Five-star reviews have 17 errors, two-star reviews have 5, and one-star reviews have 4. The model forced the three-star reviews into its two labels: 66 NEGATIVE and 34 POSITIVE. It cannot return neutral.

## Step 13 Read the mistakes


In [13]:
mistakes = graded[graded["label"] != graded["truth"]]
for review in mistakes.head(5).itertuples():
    print(review.star_rating, review.label, round(review.score, 2))
    print(review.review_body[:300], "\n")

1 POSITIVE 0.99
Don't get me wrong, the game concept is awesome. Graphics are great.  BUT  If you don't want to lose your progress when the Simcity Servers stop responding, don't buy this. There is no option to save locally just saved to their server so the smallest error will lose your work.  Overall it's a good g 

1 POSITIVE 1.0
Enjoys game 

1 POSITIVE 0.73
I wasted $7. im sure its a good game but i cant even get to the game part. i tried. a lot. 

1 POSITIVE 1.0
Dose not work right on PSVITA so was not happy but it is a fun game love twisted metal games. 

2 POSITIVE 0.87
I bought this game for my 8 year old daughter who likes the kingdom of keflings game on xbox. The graphics are very simple and it is a basic strategy/building style game. Unfortunately, it looks like it would have been cutting edge 15 to 20 years ago, but it really didn't entertain my kids for more  



**Questions:** For each of the five mistakes, explain whether the model or the customer's rating seems misleading.

1. The SimCity review praises the concept and graphics but warns that server failures lose progress. The model seems to focus too much on the praise; the 1-star rating makes sense for that serious problem.

2. "Enjoys game" sounds positive even though it has 1 star. Here the rating looks inconsistent with the written review, and the model's positive label makes sense.

3. The customer says they wasted $7 and cannot get to the game. "I'm sure it's a good game" may have confused the model. The negative rating fits the actual experience.

4. The PSVITA review says the game is fun but does not work right on that device. The model may have focused on "fun" and "love" rather than the compatibility problem.

5. The Keflings review describes a basic game that did not entertain the children. The model seems to miss the disappointment in a longer description. The low rating fits that complaint better than a positive label.

## Step 14 Predict stars


In [14]:
# Release the first model before loading another one.
del classifier
gc.collect()
star_model = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device=-1)
results = star_model(sample["review_body"].tolist(), truncation=True, batch_size=8)
sample["pred_stars"] = [int(r["label"][0]) for r in results]
print("Exact:", (sample["pred_stars"] == sample["star_rating"]).mean())
off_by = (sample["pred_stars"] - sample["star_rating"]).abs()
print("Within one star:", (off_by <= 1).mean())

The tokenizer you are loading from 'nlptown/bert-base-multilingual-uncased-sentiment' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Device set to use cpu


Exact: 0.574
Within one star: 0.912


**Questions:** How often is the exact rating right? Within one star? Which model would you choose for a game publisher's dashboard and why?

It predicted the exact rating for 57.4% of reviews and was within one star for 91.2%. For a publisher dashboard, I would choose the star model because it gives more detail and can represent the middle ratings instead of forcing everything into positive or negative. That does not mean it is 91.2% exactly correct. I would show broad trends and check examples before using it for decisions. It also printed a tokenizer warning during this run, which I would investigate before deploying it.

## Step 15 Classify 20 one star reviews by topic


In [15]:
del star_model
gc.collect()
zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1)
labels = ["price", "bugs or crashes", "account or login", "gameplay", "download or install"]
results = zsc(complaints["review_body"].tolist(), candidate_labels=labels, batch_size=4)
complaints["topic"] = [r["labels"][0] for r in results]
print(complaints["topic"].value_counts())
for review in complaints.head(3).itertuples():
    print("\nTopic:", review.topic)
    print(review.review_body)

Device set to use cpu


topic
download or install    5
bugs or crashes        5
account or login       4
price                  3
gameplay               3
Name: count, dtype: int64

Topic: price
Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get VC yea! not).

Topic: download or install
I too installed and tried to play Cobi Treasure Island. My PC does meet the proper requirements so I do not know what the problem is.

Topic: gameplay
If you liked \\"Heavy Rain\\", you'll like this.  However, if you'd like for your games to have a faster pace than 'the English Patient', you'll want to skip this title in favor of something more exciting, like \\"Parking Simulator 4\\".


**Questions:** What do customers complain about most? Print three reviews with their topics. Do the choices make sense?

Download or install and bugs or crashes tied for the most common topic, with 5 reviews each. Account or login had 4; price and gameplay had 3 each.

The three reviews printed above mostly make sense. The first mentions paying $30 extra, so price is reasonable, although it also criticizes the interface. The Cobi Treasure Island review describes installing the game and being unable to play it, so download or install is plausible, but it could also be a technical problem. The third complains that the game is slow and boring, so gameplay fits. One topic does not always capture everything in a review.

## Step 16 Change the labels


In [16]:
more_labels = labels + ["technical problems", "HR policy"]
new_results = zsc(complaints["review_body"].head(5).tolist(), candidate_labels=more_labels, batch_size=4)
for old, new in zip(results[:5], new_results):
    print("Before:", old["labels"][0], round(old["scores"][0], 2))
    print("After:", new["labels"][0], round(new["scores"][0], 2))
    print("Score sum:", round(sum(new["scores"]), 6), "\n")

Before: price 0.66
After: price 0.59
Score sum: 1.0 

Before: download or install 0.41
After: download or install 0.3
Score sum: 1.0 

Before: gameplay 0.36
After: gameplay 0.32
Score sum: 1.0 

Before: bugs or crashes 0.53
After: technical problems 0.7
Score sum: 1.0 

Before: download or install 0.79
After: download or install 0.45
Score sum: 1.0 



**Questions:** Add an overlapping label and an irrelevant label. Compare the first five reviews. What happens to labels/scores, how does the sum of 1 explain it, and what does it imply for product label choices?

I added technical problems, which overlaps bugs or crashes and download or install, and HR policy, which does not fit these game reviews. Four of the first five reviews kept their top label, but all four had lower top scores: 0.66 to 0.59, 0.41 to 0.30, 0.36 to 0.32, and 0.79 to 0.45. The fourth changed from bugs or crashes (0.53) to technical problems (0.70). HR policy was not the top label for any of the five.

The scores add up to 1 across the available labels. Adding labels changes how that total is divided, so a score can fall even though the review did not change. These are relative scores for this label set, not fixed probabilities that a complaint is true. For a real dashboard, I would use clear, reasonably distinct labels and test them on actual reviews before comparing scores.

## Run check

Restarted the kernel and ran all 16 code cells from top to bottom in Pocket. No code cells failed. The notebook keeps the actual outputs and timing from that run. Models were loaded from the local cache; the original dataset remained zipped.